# Quickstart (Google Colab): DeepSpec draft-surprise pipeline

Runs the cheap phases of the pipeline directly in Colab:

| phase | runs here? | needs |
|---|---|---|
| 0 — feasibility gates | ✅ CPU | HF access (config + tokenizer downloads) |
| 1a — WeirdChat export | ✅ CPU | `HF_TOKEN` (dataset access) |
| 1b–4 — baseline, cache, training, scoring | ❌ | a GPU node serving the 35B target |
| 5 — analysis/report | ✅ CPU | the score files from phase 4 |

**Token setup (once):** click the 🔑 **Secrets** icon in Colab's left sidebar, add a secret named
`HF_TOKEN` with your Hugging Face token, and enable *Notebook access*. If you skip this, the
token cell below simply asks you to paste the token instead.

Then just run the cells top to bottom.

In [ ]:
# 1) Clone both repositories (pulls updates on re-runs)
import os

if not os.path.exists('/content/WeirdChat'):
    !git clone --branch claude/repo-published-weights-u71yew https://github.com/Erikiss/WeirdChat /content/WeirdChat
else:
    !git -C /content/WeirdChat pull
if not os.path.exists('/content/DeepSpec'):
    !git clone --depth 1 https://github.com/Erikiss/DeepSpec /content/DeepSpec

%cd /content/WeirdChat/examples/03_deepspec_draft_surprise
os.environ['DEEPSPEC_ROOT'] = '/content/DeepSpec'

In [ ]:
# 2) Hugging Face token: Colab Secrets first, manual paste as fallback
import os

try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN loaded from Colab Secrets.')
except Exception:
    from getpass import getpass
    os.environ['HF_TOKEN'] = getpass('Paste your Hugging Face token: ')
    print('HF_TOKEN set for this session.')

In [ ]:
# 3) Install dependencies (weirdchat client + transformers; torch is preinstalled on Colab)
%pip install -q -e /content/WeirdChat transformers

# Adjust here if the target's exact HF repo id differs:
import os
os.environ['WEIRDSPEC_TARGET_MODEL'] = 'Qwen/Qwen3.6-35B-A3B'

In [ ]:
# 4) Phase 0 — feasibility gates (writes phase0_report.json)
!python phase0_feasibility.py --deepspec-root $DEEPSPEC_ROOT

import json
report = json.load(open('phase0_report.json'))
print('\nall hard gates passed:', report.get('all_hard_gates_passed'))
print('target_layer_ids:  ', report.get('recommended_target_layer_ids'))
print('draft FFN size:    ', report.get('recommended_draft_intermediate_size'),
      '(', report.get('draft_intermediate_size_source'), ')')
print('mask token:        ', report.get('recommended_mask_token'),
      report.get('recommended_mask_token_id'))
print('config nesting:    ', report.get('config_nesting'),
      '| MoE:', report.get('target_is_moe'),
      '| needs DeepSpec patch:', report.get('needs_deepspec_patch', False))
print('inserts <think>:   ', report.get('chat_template_inserts_think'),
      '| strip verified:', report.get('strip_think_verified', False))

In [ ]:
# 5) Phase 1a — export the WeirdChat qwen transcripts
#    (--max-patterns caps the download for a quick first run; drop it for the full export)
!python phase1_export_weirdchat.py --output-dir /content/data --max-patterns 25

In [ ]:
# 6) Peek at the result
from surprise_common import read_jsonl

data = read_jsonl('/content/data/weird_transcripts.jsonl')
meta = read_jsonl('/content/data/weird_meta.jsonl')
print(f'{len(data)} transcripts, {len({m["pattern_id"] for m in meta})} patterns')
print('\nexample transcript:', data[0]['id'])
for turn in data[0]['conversations'][:2]:
    print(f"  [{turn['role']}] {turn['content'][:200]}")
print('\nits metrics:', {k: meta[0][k] for k in ('behavior_id', 'elo_unexpectedness', 'match_rate')})

In [ ]:
# 7) Keep the exports: zip and download (or mount Drive instead)
!zip -qj /content/weirdspec_phase1.zip phase0_report.json
!cd /content && zip -qr weirdspec_phase1.zip data
from google.colab import files
files.download('/content/weirdspec_phase1.zip')

## Next steps (off-Colab)

Phases 1b–4 need a GPU node that serves the 35B-A3B target (DeepSpec's defaults assume 8 GPUs):

1. `phase1_baseline.sh` — baseline corpus under the WeirdChat protocol,
2. DeepSpec `prepare_target_cache.py` + `train.sh` with `dspark_qwen36_35b_a3b.py`
   (pass phase 0's `recommended_target_layer_ids` via `--opts`),
3. `phase4_score_surprise.py` on `weird_transcripts.jsonl` and `baseline_heldout.jsonl`.

Bring the two score files back to any machine (Colab works) for phase 5:
`python phase5_analyze.py --weird-scores ... --weird-meta ... --baseline-scores ... --output report.md`.
See [README.md](README.md) for details and caveats.